# Task 2.4
Before training, verify your analytical gradients from Task 2.3 against numerical finite differences. For a randomly initialised network, compute both the analytical gradient $\frac{\partial L}{\partial W_{ij}^{(1)}}$ and the finite difference approximation:

$$
\frac{\partial L}{\partial W_{ij}^{(1)}} \approx
\frac{
L\left(W_{ij}^{(1)}+\epsilon\right)
-L\left(W_{ij}^{(1)}-\epsilon\right)
}{
2\epsilon
}
\tag{8}
$$

using $\epsilon=10^{-5}$

In [91]:
# %%capture
# %run Task2.ipynb

In [92]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import math
import copy

## Equation
The two equations I have derived and will compare are:

**Gradient with respect to $W_n^{(1)}$:**

$$
\begin{aligned}
\frac{\partial L}{\partial W_n^{(1)}}
={}&
\frac{2}{N_d}\sum_{i=1}^{N_d}
\left(NN(t_i)-x(t_i)\right)
W_n^{(2)}t_i
\left(1-\tanh^2(z_{n,i})\right)
\\[6pt]
&+\lambda_{phys}\frac{2}{N_c}
\sum_{i=1}^{N_c}r_i
\Bigg[
-4W_n^{(2)}W_n^{(1)}
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
-2W_n^{(2)}(W_n^{(1)})^2t_i
\left(1-\tanh^2(z_{n,i})\right)^2
\\
&\quad
+4W_n^{(2)}(W_n^{(1)})^2t_i
\tanh^2(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{\mu}{m}
\Bigg(
W_n^{(2)}
\left(1-\tanh^2(z_{n,i})\right)
\\
&\qquad
-2W_n^{(2)}W_n^{(1)}t_i
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\Bigg)
\\
&\quad
+\frac{k}{m}W_n^{(2)}t_i
\left(1-\tanh^2(z_{n,i})\right)
\Bigg]
\\[6pt]
&+\lambda_{IC}
\Bigg[
2\left(
\sum_{j=1}^{H}W_j^{(2)}W_j^{(1)}
\left(1-\tanh^2(b_j^{(1)})\right)
\right)
W_n^{(2)}
\left(1-\tanh^2(b_n^{(1)})\right)
\Bigg]
\end{aligned}
$$

**Gradient with respect to $W_n^{(2)}$:**

$$
\begin{aligned}
\frac{\partial L}{\partial W_n^{(2)}}
={}&
\frac{2}{N_d}\sum_{i=1}^{N_d}
\left(NN(t_i)-x(t_i)\right)
\tanh(z_{n,i})
\\[6pt]
&+\lambda_{phys}\frac{2}{N_c}
\sum_{i=1}^{N_c}r_i
\Bigg[
-2(W_n^{(1)})^2
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{\mu}{m}W_n^{(1)}
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{k}{m}\tanh(z_{n,i})
\Bigg]
\\[6pt]
&+\lambda_{IC}
\Bigg[
2\left(
\sum_{j=1}^{H}W_j^{(2)}
\tanh(b_j^{(1)})+b^{(2)}-1
\right)
\tanh(b_n^{(1)})
\\
&\quad
+2\left(
\sum_{j=1}^{H}W_j^{(2)}W_j^{(1)}
\left(1-\tanh^2(b_j^{(1)})\right)
\right)
W_n^{(1)}
\left(1-\tanh^2(b_n^{(1)})\right)
\Bigg]
\end{aligned}
$$

**Loss function:**
$$
L = L_{data} + \lambda_{phys}\cdot L_{phys} + \lambda_{IC}\cdot L_{IC}
$$
**Loss function expanded:**

$$
\begin{aligned}
L ={}&
\frac{1}{N_d}\sum_{i=1}^{N_d}
\left(
NN(t_i)-e^{-2.5t_i}
\left(2\cdot0.5\cdot\cos(-0.11+22.22t_i)\right)
\right)^2
\\[6pt]
&+\lambda_{phys}\frac{1}{N_c}\sum_{i=1}^{N_c}
\left(
\frac{d^2NN}{dt^2}(t_i)
+\frac{\mu}{m}\frac{dNN}{dt}(t_i)
+\frac{k}{m}NN(t_i)
\right)^2
\\[6pt]
&+\lambda_{IC}
\left[
\left(NN(0)-1\right)^2
+\left(\frac{dNN}{dt}(0)\right)^2
\right]
\end{aligned}
$$

**Neural Network**
$$
NN(t_i) = \sum_{n=1}^{H} W_n^{(2)} \tanh\left(W_n^{(1)}t_i + b_n^{(1)}\right) + b^{(2)}
$$

**First derivative of the neural network:**

$$
\frac{dNN}{dt}(t_i) =
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
$$

**Second derivative of the neural network:**

$$
\frac{d^2NN}{dt^2}(t_i) =
-2\sum_{n=1}^{H}
W_n^{(2)}(W_n^{(1)})^2
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
$$

**Loss function with the neural network and its derivatives fully substituted:**

$$
\begin{aligned}
L ={}&
\frac{1}{N_d}\sum_{i=1}^{N_d}
\Bigg(
\sum_{n=1}^{H}
W_n^{(2)}
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
+b^{(2)}
-e^{-2.5t_i}
\left(2\cdot0.5\cdot\cos(-0.11+22.22t_i)\right)
\Bigg)^2
\\[10pt]
&+\lambda_{phys}\frac{1}{N_c}
\sum_{i=1}^{N_c}
\Bigg[
-2\sum_{n=1}^{H}
W_n^{(2)}(W_n^{(1)})^2
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
\\[6pt]
&\qquad
+\frac{\mu}{m}
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
\\[6pt]
&\qquad
+\frac{k}{m}
\left(
\sum_{n=1}^{H}
W_n^{(2)}
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
+b^{(2)}
\right)
\Bigg]^2
\\[10pt]
&+\lambda_{IC}
\Bigg[
\left(
\sum_{n=1}^{H}
W_n^{(2)}\tanh\left(b_n^{(1)}\right)
+b^{(2)}-1
\right)^2
\\[6pt]
&\qquad
+\left(
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(b_n^{(1)}\right)
\right)
\right)^2
\Bigg]
\end{aligned}
$$

Firstly, creating the neural network

In [93]:
#constants
m = 1.0 # mass
mu = 5 # friction constant
k = 500 # spring constant

omega_intial = np.sqrt(k)
delta = mu/(2*m)
omega = np.sqrt(omega_intial**2 - delta**2)

def A(phi):
    return 1 / (2*math.cos(phi))

phi = np.arctan(-1*delta/omega)

t_train = torch.linspace(start = 0, end = 0.5, steps = 10)
t_train = t_train.view(-1,1) # We transform it into shape (samples, features), instead of (samples). In our case that is (10,1)

def x(t):
    return  np.exp(-1*delta*t) *(2*A(phi)* np.cos(omega*t + phi))
y_train = x(t_train)

C:\Users\Saads\AppData\Local\Temp\ipykernel_22760\4019892306.py:19: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)



In [94]:
t_train = t_train.reshape(-1, 1)
t_train = t_train.detach().clone().requires_grad_(True).double()
t0 = torch.tensor([0.0], requires_grad=True)
# Creating the model
model = nn.Sequential(nn.Linear(1, 10), nn.Tanh(), nn.Linear(10, 1))
model = model.double()

# Making NN a function of t and then deriving it
# def NN(t):
#     return model(t)
# NN = model(t_train)
# dNN_dt = torch.autograd.grad(NN, t_train, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
# dNN2_dt2 = torch.autograd.grad(dNN_dt, t_train, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

# # Physics loss. The residuals must equal 0
# rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
# loss_phys = torch.mean(rt**2)

# # Initial conditions loss. x(0) = 1 and x'(0) = 0
# NN_0 = model(t0)
# dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
# loss_IC1 = torch.mean((NN_0 - 1)**2)
# loss_IC2 = torch.mean(dNN_0**2)

# # Data loss. The model must provide the same output as the closed form solution
# # Version 1
# # loss_data = []
# # for i in range(len(t_train)):
# #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
# #     loss_data.append(data_loss_increment)
# # loss_data = torch.stack(loss_data).mean()
# # Version 2
# predictions = model(t_train)
# loss_fn = nn.MSELoss()
# assert predictions.shape == y_train.shape
# loss_data = loss_fn(predictions, y_train)


In [95]:
def calc_loss():
    t_col = torch.linspace(start = 0, end = 1, steps = 50).reshape(-1, 1).detach().clone().requires_grad_(True).double()
    NN = model(t_col)
    dNN_dt = torch.autograd.grad(NN, t_col, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
    dNN2_dt2 = torch.autograd.grad(dNN_dt, t_col, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

    # Physics loss. The residuals must equal 0
    rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
    loss_phys = torch.mean(rt**2)

    # Initial conditions loss. x(0) = 1 and x'(0) = 0
    t0 = torch.tensor([0.0], requires_grad=True).double()
    NN_0 = model(t0)
    dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
    loss_IC1 = torch.mean((NN_0 - 1)**2)
    loss_IC2 = torch.mean(dNN_0**2)

    # Data loss. The model must provide the same output as the closed form solution
    # Version 1
    # loss_data = []
    # for i in range(len(t_train)):
    #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
    #     loss_data.append(data_loss_increment)
    # loss_data = torch.stack(loss_data).mean()
    # Version 2
    predictions = model(t_train)
    loss_fn = nn.MSELoss()
    
    assert predictions.shape == y_train.shape
    loss_data = loss_fn(predictions, y_train)
    
    lam_phys = 1e-4
    lam_ic = 1

    total_loss = loss_data + loss_phys * lam_phys + (loss_IC1 + loss_IC2) * lam_ic

    return total_loss


In [96]:
# print("Total loss shape: ", loss.shape)
# print("Total loss: ", loss.item())
# for i in range(10):
#     print(f'w({i}):', model[0].weight[i,0].item())
# Finite difference approximation

# w1_plus = model[0].weight.tolist()
# w1_minus = model[0].weight.tolist()
# b1 = model[0].bias.tolist()
# w2 = model[2].weight[0].tolist()
# b2 = model[2].bias.item()
e = 1e-5
fda_list_w1 = []
fda_list_w2 = []
l_plus_w1 = []
l_minus_w1 = []
l_plus_w2 = []
l_minus_w2 = []
loss_list_plus = []
loss_list_minus = []

for i in range(10):
    og_weight = model[0].weight[i,0].item()
    with torch.no_grad():
        model[0].weight[i, 0] = og_weight + e
    l_plus_w1.append(calc_loss().item())

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight - e
    l_minus_w1.append(calc_loss().item())

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight

    fda_list_w1.append((l_plus_w1[-1] - l_minus_w1[-1]) / (2 * e))

for i in range(10):
    og_weight = model[2].weight[0, i].item()
    with torch.no_grad():
        model[2].weight[0, i] = og_weight + e
    l_plus_w2.append(calc_loss().item())

    with torch.no_grad():
        model[2].weight[0, i] = og_weight

    with torch.no_grad():
        model[2].weight[0, i] = og_weight - e
    l_minus_w2.append(calc_loss().item())

    with torch.no_grad():
        model[2].weight[0, i] = og_weight

    fda_list_w2.append((l_plus_w2[-1] - l_minus_w2[-1]) / (2 * e))

print("W1 Plus losses:", l_plus_w1)
print("W1 Minus losses:", l_minus_w1)
print("W2 Plus losses:", l_plus_w2)
print("W2 Minus losses:", l_minus_w2)
print("W1 Numerical gradient: ", fda_list_w1)
print("W2 Numerical gradient: ", fda_list_w2)

W1 Plus losses: [9.720330323976267, 9.720324554001122, 9.720281362850235, 9.720271160708364, 9.720287169743608, 9.720259317825239, 9.720287511318146, 9.720306230927982, 9.720319464405161, 9.720315998243965]
W1 Minus losses: [9.72025648683939, 9.720262256939307, 9.720305447871025, 9.720315650041119, 9.72029964115024, 9.720327492925028, 9.7202992995447, 9.720280579891991, 9.720267346426375, 9.720270812432732]
W2 Plus losses: [9.720408977772829, 9.720287071310615, 9.720053958612244, 9.720213033163946, 9.720537562917787, 9.720229593365016, 9.72042264272438, 9.720346634098131, 9.720333907112424, 9.720501054427466]
W2 Minus losses: [9.720177834437647, 9.720299739640058, 9.720532855651495, 9.720373778346824, 9.720049251401415, 9.72035721805079, 9.720164169446134, 9.720240176984362, 9.720252903824873, 9.720085758919629]
W1 Numerical gradient:  [3.6918568438792927, 3.1148530907465495, -1.2042510395104955, -2.224466637734679, -0.623570331548251, -3.4087549894756815, -0.5894113277449264, 1.282551

In [97]:
model.zero_grad()

total_loss = calc_loss()
total_loss.backward()

dW1 = model[0].weight.grad[:, 0].tolist()  # 10 first-layer gradients
dW2 = model[2].weight.grad[0, :].tolist()  # 10 second-layer gradients
print("dw1: ", dW1)
print("dw2: ", dW2)

dw1:  [3.691856844029676, 3.114853090909364, -1.2042510396780157, -2.2244666376538778, -0.6235703314604261, -3.4087549896091307, -0.589411327778594, 1.2825517995467903, 2.6058989393874192, 2.2592905616018992]
dw2:  [11.557166759086776, -0.6334164721761197, -23.944851962574017, -8.037259143830232, 24.415575818666372, -6.381234288612083, 12.92366391234513, 5.322855688446603, 4.050164377660552, 20.764775391731405]


In [122]:
dW1 = np.asarray(dW1)
dW2 = np.asarray(dW2)
fda_list_w1 = np.asarray(fda_list_w1)
fda_list_w2 = np.asarray(fda_list_w2)

absolute_error_w1 = np.abs(dW1 - fda_list_w1)
absolute_error_w2 = np.abs(dW2 - fda_list_w2)
fig = px.line(y = [absolute_error_w1, absolute_error_w2], title='Absolute gradient error')
fig.update_layout(xaxis_title='Weight index', yaxis_title='Absolute error', template='plotly_dark', 
                 showlegend=True)
fig.show()

# 2.5

In [99]:
model_copy1 = copy.deepcopy(model)
model_copy2 = copy.deepcopy(model)

In [100]:

# data points and collocation points
t_data = np.linspace(0, 0.5, 10)
x_data = x(t_data)
t_col = np.linspace(0, 1, 50)


lam_ic = 1

def NN_manual(t, W1, b1, W2, b2):
    return np.sum(W2 * np.tanh(W1*t + b1)) + b2

def dNN(t, W1, b1, W2, b2):
    return np.sum(W2 * W1 * (1 - np.tanh(W1*t + b1)**2))

def ddNN(t, W1, b1, W2, b2):
    tanh = np.tanh(W1*t + b1)
    return np.sum(-2 * W2 * W1**2 * tanh * (1 - tanh**2))

def loss(W1, b1, W2, b2, lam_phys):
  # data term
  L_data = 0
  for i in range(len(t_data)):
      L_data += (NN_manual(t_data[i], W1, b1, W2, b2) - x_data[i])**2
  L_data = L_data / len(t_data)

  # physics term
  L_phys = 0
  for i in range(len(t_col)):
      t = t_col[i]
      r = ddNN(t, W1, b1, W2, b2) + mu/m * dNN(t, W1, b1, W2, b2) + k/m * NN_manual(t, W1, b1, W2, b2)
      L_phys += r**2
  L_phys = L_phys / len(t_col)

  # initial conditions
  L_ic = (NN_manual(0, W1, b1, W2, b2) - 1)**2 + (dNN(0, W1, b1, W2, b2) - 0)**2

  return L_data + lam_phys*L_phys + lam_ic*L_ic

def gradients(W1, b1, W2, b2, lam_phys):
    dW1 = np.zeros(len(W1))
    db1 = np.zeros(len(b1))
    dW2 = np.zeros(len(W2))
    db2 = 0

    # data term
    for i in range(len(t_data)):
        t = t_data[i]
        tanh = np.tanh(W1*t + b1)
        e = NN_manual(t, W1, b1, W2, b2) - x_data[i]

        dW1 += 2/len(t_data) * e * W2 * (1 - tanh**2) * t
        db1 += 2/len(t_data) * e * W2 * (1 - tanh**2)
        dW2 += 2/len(t_data) * e * tanh
        db2 += 2/len(t_data) * e

    # physics term
    for i in range(len(t_col)):
        t = t_col[i]
        tanh = np.tanh(W1*t + b1)
        r = ddNN(t, W1, b1, W2, b2) + mu/m * dNN(t, W1, b1, W2, b2) + k/m * NN_manual(t, W1, b1, W2, b2)

        # eq (42)
        dr_dW1 = (-4 * W2 * W1 * tanh * (1 - tanh**2)
                  - 2 * W2 * W1**2 * t * (1 - tanh**2)**2
                  + 4 * W2 * W1**2 * t * tanh**2 * (1 - tanh**2)
                  + mu/m * (W2 * (1 - tanh**2) - 2 * W2 * W1 * t * tanh * (1 - tanh**2))
                  + k/m * W2 * t * (1 - tanh**2))

        # eq (43)
        dr_dW2 = (-2 * W1**2 * tanh * (1 - tanh**2)
                  + mu/m * W1 * (1 - tanh**2)
                  + k/m * tanh)

        # biases: same as for W1 but without the factor t from the inner derivative
        dr_db1 = (-2 * W2 * W1**2 * (1 - tanh**2)**2
                  + 4 * W2 * W1**2 * tanh**2 * (1 - tanh**2)
                  + mu/m * (-2 * W2 * W1 * tanh * (1 - tanh**2))
                  + k/m * W2 * (1 - tanh**2))
        dr_db2 = k/m

        dW1 += lam_phys * 2/len(t_col) * r * dr_dW1
        db1 += lam_phys * 2/len(t_col) * r * dr_db1
        dW2 += lam_phys * 2/len(t_col) * r * dr_dW2
        db2 += lam_phys * 2/len(t_col) * r * dr_db2

    # initial conditions (t = 0)
    tanh0 = np.tanh(b1)
    NN0 = NN_manual(0, W1, b1, W2, b2)
    dNN0 = dNN(0, W1, b1, W2, b2)

    dW1 += lam_ic * 2 * dNN0 * W2 * (1 - tanh0**2)
    db1 += lam_ic * (2 * (NN0 - 1) * W2 * (1 - tanh0**2) - 4 * dNN0 * W2 * W1 * tanh0 * (1 - tanh0**2))
    dW2 += lam_ic * (2 * (NN0 - 1) * tanh0 + 2 * dNN0 * W1 * (1 - tanh0**2))
    db2 += lam_ic * 2 * (NN0 - 1)

    return dW1, db1, dW2, db2

In [101]:
def calc_loss(lam_phys):
    t_col = torch.linspace(start = 0, end = 1, steps = 50).reshape(-1, 1).detach().clone().requires_grad_(True).double()
    NN = model_copy2(t_col)
    dNN_dt = torch.autograd.grad(NN, t_col, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
    dNN2_dt2 = torch.autograd.grad(dNN_dt, t_col, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

    # Physics loss. The residuals must equal 0
    rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
    loss_phys = torch.mean(rt**2)

    # Initial conditions loss. x(0) = 1 and x'(0) = 0
    t0 = torch.tensor([0.0], requires_grad=True).double()
    NN_0 = model_copy2(t0)
    dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
    loss_IC1 = torch.mean((NN_0 - 1)**2)
    loss_IC2 = torch.mean(dNN_0**2)

    # Data loss. The model must provide the same output as the closed form solution
    # Version 1
    # loss_data = []
    # for i in range(len(t_train)):
    #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
    #     loss_data.append(data_loss_increment)
    # loss_data = torch.stack(loss_data).mean()
    # Version 2
    predictions = model_copy2(t_train)
    loss_fn = nn.MSELoss()
    
    assert predictions.shape == y_train.shape
    loss_data = loss_fn(predictions, y_train)
    
    lam_ic = 1

    total_loss = loss_data + loss_phys * lam_phys + (loss_IC1 + loss_IC2) * lam_ic

    return total_loss


In [102]:
W1 = model_copy1[0].weight.detach().numpy()[:, 0]
b1 = model_copy1[0].bias.detach().numpy()
W2 = model_copy1[2].weight.detach().numpy()[0, :]
b2 = model_copy1[2].bias.detach().numpy()[0]

dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2, lam_phys=1e-4)
manual_grads = np.concatenate([
    dW1.flatten(),
    db1.flatten(),
    dW2.flatten(),
    np.atleast_1d(db2)
])

model.zero_grad()
autograd_loss = calc_loss(lam_phys=1e-4)
autograd_tensors = torch.autograd.grad(autograd_loss, model_copy2.parameters())
autograd_grads = np.concatenate([gradient.detach().numpy().flatten() for gradient in autograd_tensors])

relative_error = np.linalg.norm(manual_grads - autograd_grads) / np.linalg.norm(manual_grads)
print(relative_error)

3.134091852174757e-09


In [103]:
model_copy1 = copy.deepcopy(model)
model_copy2 = copy.deepcopy(model)
model_copy3 = copy.deepcopy(model)

In [104]:
epoch = 20000
optimizer = torch.optim.Adam(model_copy1.parameters(), lr = 0.01)
losses = []

for i in range(epoch):
  # Loss function  where the forward pass also happens
  W1 = model_copy1[0].weight.detach().numpy()[:, 0]
  b1 = model_copy1[0].bias.detach().numpy()
  W2 = model_copy1[2].weight.detach().numpy()[0, :]
  b2 = model_copy1[2].bias.detach().numpy()[0]

  losses.append(loss(W1, b1, W2, b2, lam_phys=1e-4))

  # Update weights
  dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2, lam_phys=1e-4)
  model_copy1[0].weight.grad = torch.tensor(dW1).reshape(10, 1)
  model_copy1[0].bias.grad = torch.tensor(db1)
  model_copy1[2].weight.grad = torch.tensor(dW2).reshape(1, 10)
  model_copy1[2].bias.grad = torch.tensor([db2], dtype=torch.float64)

  optimizer.step()

  if i % 1000 == 0:
      print(i, losses[-1])

0 9.72029347056163
1000 1.0877739964542428
2000 0.9500063176078827
3000 0.38431319570338424
4000 0.23540524607412627
5000 0.20131899917762514
6000 0.15389018110171487
7000 0.1055197520098478
8000 0.07892722655953102
9000 0.06815151818170753
10000 0.06269786818616842
11000 0.057454866242662724
12000 0.05240186436253292
13000 0.04973679890062303
14000 0.04791009390977439
15000 0.04654350943806349
16000 0.04545042327117228
17000 0.05229405682843391
18000 0.044102957869580156
19000 0.04314696593662918


## Changing regularization

In [ ]:
epoch = 20000
lambda_values = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
models = [copy.deepcopy(model) for k in lambda_values]
optimizers = [torch.optim.Adam(m.parameters(), lr=0.01) for m in models]
loss_histories = [[] for k in lambda_values]

for i in range(epoch):
  for j, lam in enumerate(lambda_values):
    current_model = models[j]
    current_optimzer = optimizers[j]
    W1 = current_model[0].weight.detach().numpy()[:, 0]
    b1 = current_model[0].bias.detach().numpy()
    W2 = current_model[2].weight.detach().numpy()[0, :]
    b2 = current_model[2].bias.detach().numpy()[0]
  
    loss_histories[j].append(loss(W1, b1, W2, b2, lam))
  
    # Update weights
    dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2, lam)
    current_model[0].weight.grad = torch.tensor(dW1).reshape(10, 1)
    current_model[0].bias.grad = torch.tensor(db1)
    current_model[2].weight.grad = torch.tensor(dW2).reshape(1, 10)
    current_model[2].bias.grad = torch.tensor([db2], dtype=torch.float64)
  
    current_optimzer.step()
fig = go.Figure()
for lam, reg_loss in zip(lambda_values, loss_histories):
  fig.add_scatter(x=list(range(len(reg_loss))), y=reg_loss, mode='lines', name=f'lam={lam:g}')
fig.update_layout(title='Different regularizations', xaxis_title="Epoch", yaxis_title="Losses", template="plotly_dark")
fig.show()

In [106]:
t_eval = torch.linspace(0, 1, 100).reshape(-1,1).double()
reg_predictions = [[] for k in lambda_values]
with torch.no_grad():
    for i in range(len(lambda_values)):
        reg_predictions[i] = models[i](t_eval).numpy().flatten()

fig = go.Figure()
for lam, reg_pred in zip(lambda_values, reg_predictions):
  fig.add_scatter(x=t_eval.flatten(), y=reg_pred, mode='lines', name=f'lam={lam:g}')
fig.update_layout(title='Inference of different regularization', xaxis_title="Time", yaxis_title="Diplacement", template="plotly_dark")
fig.show()

# Task 2.6

In [107]:
loss_list = []
optimizer = torch.optim.Adam(model_copy2.parameters(), lr = 0.01)
for i in range(epoch):
    model_copy2.zero_grad()
    grad_loss = (calc_loss(lam_phys=1e-4))
    loss_list.append(grad_loss.item())
    grads = torch.autograd.grad(grad_loss, model_copy2.parameters())
    for parameter, gradient in zip(model_copy2.parameters(), grads):
        parameter.grad = gradient
    optimizer.step()
    if i % 1000 == 0:
        print(i, loss_list[-1])

0 9.720293405416225
1000 1.0877739725628808
2000 0.9500063014961468
3000 0.3849924306323802
4000 0.23037921289683422
5000 0.20065406055513896
6000 0.1618364617911654
7000 0.10401820800280885
8000 0.07936918840498194
9000 0.06810992159711991
10000 0.0674801474426738
11000 0.057261427110014945
12000 0.05233617679897129
13000 0.04965541011822625
14000 0.04788364253421906
15000 0.046540378100057
16000 0.04546443419833791
17000 0.04457560560368013
18000 0.043819032708722706
19000 0.04315710960826222


In [108]:
loss_array = np.asarray(loss_list)
losses_array = np.asarray(losses)

In [119]:
abs_differnce_losses = np.abs(losses_array - loss_array)

fig = px.line(y = abs_differnce_losses, title='Absolute error losses')
fig.update_layout(xaxis_title='Index', yaxis_title='Absolute error', template='plotly_dark', 
                 showlegend=True)
fig.show()

In [120]:
fig = px.line(y=[losses_array, loss_array], title="Losses comparison")
fig.update_layout(xaxis_title='Index', yaxis_title='Lossesr', template='plotly_dark', 
                 showlegend=True)
fig.show()

In [121]:
t_eval = torch.linspace(0, 1, 100).reshape(-1,1).double()
with torch.no_grad():
    manual_predictions = model_copy1(t_eval).numpy().flatten()
    autograd_predictions = model_copy2(t_eval).numpy().flatten()
fig = px.line(x=t_eval.flatten(), y=[manual_predictions, autograd_predictions], title='Inference comparison')
fig.update_layout(xaxis_title='Time', yaxis_title='Displacement', template='plotly_dark', 
                 showlegend=True)
fig.show()

# Task 2.7

In [112]:
model_hard = copy.deepcopy(model)
def x_0(t):
    return 1 + t**2 * model_hard(t)

In [113]:
t = torch.tensor([0.0], requires_grad=True, dtype=torch.float64)
value = x_0(t).backward()
print(x_0(t).item())
print(t.grad.item())

1.0
0.0


In [114]:
def calc_loss_hard(lam_phys):
    t_col = torch.linspace(start = 0, end = 1, steps = 50).reshape(-1, 1).detach().clone().requires_grad_(True).double()
    NN = x_0(t_col)
    dNN_dt = torch.autograd.grad(NN, t_col, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
    dNN2_dt2 = torch.autograd.grad(dNN_dt, t_col, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

    # Physics loss. The residuals must equal 0
    rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
    loss_phys = torch.mean(rt**2)

    # Initial conditions loss. x(0) = 1 and x'(0) = 0


    # Data loss. The model must provide the same output as the closed form solution
    # Version 1
    # loss_data = []
    # for i in range(len(t_train)):
    #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
    #     loss_data.append(data_loss_increment)
    # loss_data = torch.stack(loss_data).mean()
    # Version 2
    predictions = x_0(t_train)
    loss_fn = nn.MSELoss()
    
    assert predictions.shape == y_train.shape
    loss_data = loss_fn(predictions, y_train)
    
    lam_ic = 1

    total_loss = loss_data + lam_phys * loss_phys

    return total_loss


In [115]:
loss_list_hard = []
optimizer = torch.optim.Adam(model_hard.parameters(), lr = 0.01)
for i in range(epoch):
    model_hard.zero_grad()
    grad_loss_hard = (calc_loss_hard(lam_phys=1e-4))
    loss_list_hard.append(grad_loss_hard.item())
    grads = torch.autograd.grad(grad_loss_hard, model_hard.parameters())
    for parameter, gradient in zip(model_hard.parameters(), grads):
        parameter.grad = gradient
    optimizer.step()
    if i % 1000 == 0:
        print(i, loss_list_hard[-1])

0 39.814730055287974
1000 3.373973774554705
2000 2.332808740881068
3000 1.7362881868535707
4000 1.3166463907782222
5000 0.9514160396975495
6000 0.6833334984699462
7000 0.5077295203498027
8000 0.420691080499054
9000 0.3664528336348652
10000 0.3277452872664684
11000 0.29821973687678627
12000 0.27500479068802725
13000 0.25599566326629974
14000 0.2404816695396887
15000 0.22748219784472906
16000 0.21637673264601157
17000 0.20678079600540705
18000 0.19834307974512383
19000 0.19102968417694624


In [116]:
with torch.no_grad():
    hard_predictions = x_0(t_eval).flatten()
    soft_pred = model_copy2(t_eval).flatten()
    analytical_predictions = x(t_eval).flatten()
fig = go.Figure()
fig.add_scatter(x=t_eval.flatten(), y=hard_predictions, mode="lines", name="Hard enforcement")
fig.add_scatter(x=t_eval.flatten(), y=analytical_predictions, mode="lines", name="Analytical")
fig.add_scatter(x=t_eval.flatten(), y=soft_pred, mode='lines', name='Soft Initial conditions')
fig.update_layout(
    title="Inference",
    xaxis_title="Time",
    yaxis_title="Displacement",
    template="plotly_dark"
)
fig.show()

C:\Users\Saads\AppData\Local\Temp\ipykernel_22760\4019892306.py:19: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)



In [117]:
fig = go.Figure()
fig.add_scatter(y=loss_list, mode='lines', name='Soft Initial conditions')
fig.add_scatter(y=loss_list_hard, mode='lines', name='Hard Initial conditions')
fig.update_layout(
    title="Losses",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    template="plotly_dark"
)
fig.show()

In [118]:
mse_hard = torch.mean((hard_predictions - analytical_predictions)**2)
mse_soft = torch.mean((soft_pred - analytical_predictions)**2)

print("Hard IC MSE:", mse_hard.item())
print("Soft IC MSE:", mse_soft.item())

Hard IC MSE: 0.022670290972718582
Soft IC MSE: 0.011010877080021011


the soft initial-condition PINN achieved better overall prediction accuracy than the hard-constraint PINN in this experimen